In [ ]:
import marimo as mo

In [ ]:
import sys

# sys.path.insert(0, "/tmp/mmdet-venv/lib/python3.11/site-packages")
import torch
import os
import subprocess

In [ ]:
os.environ["HF_HUB_DISABLE_XET"] = "1"

from pathlib import Path as _HfPath
from huggingface_hub import snapshot_download

HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
os.environ["HF_TOKEN"] = HF_TOKEN

# Dataset markers only. preprocess_data.py belongs to the code/preprocess flow,
# so it must not decide whether LevirShipData itself is already present.
_data_required_paths = [
    _HfPath("All Images"),
    _HfPath("All Annotations"),
]
_image_stems = {p.stem for p in _HfPath("All Images").glob("*.png")}
_label_stems = {p.stem for p in _HfPath("All Annotations").glob("*.txt")}
_data_ready = (
    len(_image_stems) == 3896
    and len(_label_stems) == 3896
    and _image_stems == _label_stems
)

if _data_ready:
    print("SKIP LevirShipData download: data folders already exist.")
else:
    print("Downloading LevirShipData from Hugging Face...")
    snapshot_download(
        repo_id="duyle2408/LevirShipData",
        repo_type="dataset",
        token=HF_TOKEN,
        local_dir=".",
        max_workers=2,
        # local_dir_use_symlinks=False,
    )
    print("Done downloading LevirShipData")

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "albumentations",
        "kornia",
        "timm",
        "huggingface_hub",
        "opencv-python-headless",
    ],
    check=True,
)

In [ ]:
# ================================================================
# GITHUB TOKEN — live-only cell.
# Dán token vào đây nếu repo GitHub private hoặc cần tránh auth prompt.
# Không push/export notebook public khi cell này chứa token thật.
# ================================================================
import os as _yolo_token_os

# TODO: paste token here in live if needed, e.g. "github_pat_...".
yolo_github_token = "[REDACTED_SECRET]"

if yolo_github_token:
    _yolo_token_os.environ["GITHUB_TOKEN"] = yolo_github_token

print("GITHUB_TOKEN configured for GitHub clone/pull:", bool(_yolo_token_os.environ.get("GITHUB_TOKEN")))

In [ ]:
# ================================================================
# LOAD YOLO FTSC OPTIMIZER/PROVENANCE AUDIT CODEBASE FROM GITHUB — full repo sync.
# Dataset vẫn lấy từ cell LevirShipData phía trên.
# ================================================================
from pathlib import Path as _YoloCodePath
import os as _yolo_code_os
import shutil as _yolo_code_shutil
import subprocess as _yolo_code_subprocess
import time as _yolo_code_time
from urllib.parse import urlparse as _yolo_urlparse, urlunparse as _yolo_urlunparse


yolo_github_repo_url = "https://github.com/TLMHoang-AI/YOLO_FTSC.git"
yolo_code_dir = _YoloCodePath("/marimo/yolo_code")
yolo_github_branch = "main"


def _yolo_tokenized_url(_url: str) -> str:
    _token = _yolo_code_os.environ.get("GITHUB_TOKEN")
    if not _token or not _url.startswith("https://github.com/"):
        return _url
    _parsed = _yolo_urlparse(_url)
    return _yolo_urlunparse(
        _parsed._replace(netloc=f"x-access-token:{_token}@{_parsed.netloc}")
    )


def _yolo_redact(_parts):
    _token = _yolo_code_os.environ.get("GITHUB_TOKEN", "")
    if not _token:
        return _parts
    return [_part.replace(_token, "<GITHUB_TOKEN>") for _part in _parts]


def _yolo_git(_args, *, _cwd=None, _check=True):
    _env = dict(_yolo_code_os.environ)
    _env["GIT_TERMINAL_PROMPT"] = "0"
    _result = _yolo_code_subprocess.run(
        ["git", *_args],
        cwd=_cwd,
        env=_env,
        text=True,
        capture_output=True,
    )
    if _result.stdout:
        print(_result.stdout)
    if _result.stderr:
        print(_result.stderr)
    if _check and _result.returncode != 0:
        _safe = " ".join(_yolo_redact(["git", *_args]))
        raise RuntimeError(f"Git command failed: {_safe}")
    return _result


# _yolo_core_required_markers = [
#     yolo_code_dir
#     / "train_levir_scripts/train_all_levir_yolov8n_p2_ftsc_optimizer_provenance_audit.py",
#     yolo_code_dir / "train_levir_scripts/test_ftsc_optimizer_provenance_audit.py",
#     yolo_code_dir / "models_related/ultralytics/ultralytics/engine/trainer.py",
#     yolo_code_dir / "models_related/ultralytics/ultralytics/nn/modules/ftsc.py",
#     yolo_code_dir / "models_related/ultralytics/ultralytics/utils/loss.py",
#     yolo_code_dir
#     / "models_related/models_config/yolov8/levir/yolov8n_p2_levir_ftsc_y0_baseline.yaml",
#     yolo_code_dir
#     / "models_related/models_config/yolov8/levir/yolov8n_p2_levir_ftsc_af_y4_f5_position_dflcls.yaml",
#     yolo_code_dir
#     / "models_related/models_config/yolov8/levir/yolov8n_p2_levir_ftsc_af_y4_f5_position_dflcls_fixed1.yaml",
# ]


# def _yolo_missing_markers():
#     return [str(_p) for _p in _yolo_core_required_markers if not _p.exists()]


_repo_url_for_clone = _yolo_tokenized_url(yolo_github_repo_url)

if (yolo_code_dir / ".git").is_dir():
    print("Existing yolo_code git repo found; forcing full GitHub sync...")
    _yolo_git(["remote", "set-url", "origin", _repo_url_for_clone], _cwd=yolo_code_dir)
    _yolo_git(["fetch", "origin", yolo_github_branch], _cwd=yolo_code_dir)
    # Intentional: the live experiment checkout must mirror the pushed commit.
    _yolo_git(["reset", "--hard", f"origin/{yolo_github_branch}"], _cwd=yolo_code_dir)
    _yolo_git(
        ["clean", "-fd", "-e", "runs/", "-e", "datasets/", "-e", "yolov8n.pt", "-e", "yolov9t.pt", "-e", "yolov10n.pt", "-e", "yolo11n.pt"],
        _cwd=yolo_code_dir,
    )
    _yolo_git(["remote", "set-url", "origin", yolo_github_repo_url], _cwd=yolo_code_dir)
elif yolo_code_dir.exists():
    _backup = yolo_code_dir.with_name(
        f"{yolo_code_dir.name}_partial_backup_{int(_yolo_code_time.time())}"
    )
    print(f"Found non-git/partial yolo_code. Moving it to backup: {_backup}")
    _yolo_code_shutil.move(str(yolo_code_dir), str(_backup))
    print("Cloning FULL YOLO_FTSC repo from GitHub...")
    _yolo_git([
        "clone",
        "--branch",
        yolo_github_branch,
        _repo_url_for_clone,
        str(yolo_code_dir),
    ])
    _yolo_git(["remote", "set-url", "origin", yolo_github_repo_url], _cwd=yolo_code_dir)
else:
    print("Cloning FULL YOLO_FTSC repo from GitHub...")
    _yolo_git([
        "clone",
        "--branch",
        yolo_github_branch,
        _repo_url_for_clone,
        str(yolo_code_dir),
    ])
    _yolo_git(["remote", "set-url", "origin", yolo_github_repo_url], _cwd=yolo_code_dir)

# _missing_yolo = _yolo_missing_markers()
# if _missing_yolo:
#     raise FileNotFoundError(
#         "YOLO FTSC full repo sync finished but required files are still missing:\n- "
#         + "\n- ".join(_missing_yolo)
#     )

print("YOLO FTSC optimizer/provenance audit codebase ready:", yolo_code_dir)
print("Latest commit:")
_yolo_git(["log", "--oneline", "-1"], _cwd=yolo_code_dir)

In [ ]:
# Materialize the official Ultralytics YOLOv8n initialization checkpoint.
# The repository sync cell preserves this file across git clean/reset.
from pathlib import Path as _FtscPretrainedPath
import sys as _ftsc_pretrained_sys
_ftsc_pretrained_vendor = (
    _FtscPretrainedPath(yolo_code_dir) / "models_related" / "ultralytics"
).resolve()
if not (_ftsc_pretrained_vendor / "ultralytics" / "__init__.py").is_file():
    raise FileNotFoundError("Vendored FTSC Ultralytics fork missing after GitHub sync")
_ftsc_pretrained_vendor_path = str(_ftsc_pretrained_vendor)
_ftsc_pretrained_sys.path[:] = [
    path for path in _ftsc_pretrained_sys.path if path != _ftsc_pretrained_vendor_path
]
_ftsc_pretrained_sys.path.insert(0, _ftsc_pretrained_vendor_path)
import ultralytics as _ftsc_pretrained_module
if _ftsc_pretrained_vendor not in _FtscPretrainedPath(_ftsc_pretrained_module.__file__).resolve().parents:
    raise ImportError("Imported Ultralytics is not the GitHub-synced FTSC fork; restart the kernel")
from ultralytics.utils.downloads import (
    attempt_download_asset as _ftsc_attempt_download_asset,
)


yolo_ftsc_pretrained_path = (
    _FtscPretrainedPath(yolo_code_dir) / "yolov8n.pt"
).resolve()

if not yolo_ftsc_pretrained_path.is_file():
    _ftsc_downloaded_pretrained = _FtscPretrainedPath(
        _ftsc_attempt_download_asset(yolo_ftsc_pretrained_path)
    ).resolve()
    if _ftsc_downloaded_pretrained != yolo_ftsc_pretrained_path:
        raise RuntimeError(
            "Ultralytics downloaded the checkpoint to an unexpected path: "
            f"{_ftsc_downloaded_pretrained}"
        )

if not yolo_ftsc_pretrained_path.is_file():
    raise FileNotFoundError(
        "Official Ultralytics yolov8n.pt download did not produce the expected file: "
        f"{yolo_ftsc_pretrained_path}"
    )

print(
    "Official Ultralytics YOLOv8n checkpoint ready: "
    f"{yolo_ftsc_pretrained_path} "
    f"({yolo_ftsc_pretrained_path.stat().st_size:,} bytes)"
)
mo.status.toast("Official YOLOv8n pretrained checkpoint is ready", kind="success")
yolo_ftsc_pretrained_path

In [ ]:
# Historical-H2-matched FTSC Position Alpha Sensitivity (isolated, disabled by default).
# Run AFTER the GitHub sync and yolov8n pretrained cells have completed.
RUN_LEVIR_FTSC_ALPHA = False
LEVIR_FTSC_ALPHA_PREFLIGHT_ONLY = True  # Set False explicitly to allow training
LEVIR_FTSC_ALPHA_SELECTION = {
    "cases": ["ALPHA4_rerun", "ALPHA8_rerun"],  # α6 is historical H2 control
    "seeds": [42, 43, 44],
    "workers": 20,
    "device": "cuda",
    "allow_alpha6_bridge": False,
}
if RUN_LEVIR_FTSC_ALPHA:
    from pathlib import Path as _AlphaPath
    import sys as _alpha_sys
    import importlib as _alpha_importlib
    import importlib as _alpha_importlib

    _alpha_root = _AlphaPath(yolo_code_dir).resolve()
    _alpha_vendor = _alpha_root / "models_related" / "ultralytics"
    if not (_alpha_root / ".git").is_dir():
        raise RuntimeError("Alpha suite requires a synced Git checkout")
    for _alpha_import_root in (str(_alpha_root), str(_alpha_vendor)):
        if _alpha_import_root not in _alpha_sys.path:
            _alpha_sys.path.insert(0, _alpha_import_root)
    # The notebook's existing import cell already selects the local FTSC fork.
    from train_levir_scripts import train_levir_ftsc_position_alpha_sensitivity_v1 as _alpha_suite
    _alpha_suite = _alpha_importlib.reload(_alpha_suite)
    from utils.ftsc_results_uploader import FTSCResultsUploader as _AlphaUploader

    _alpha_args = _alpha_suite.parse_args([])
    _alpha_args.cases = list(LEVIR_FTSC_ALPHA_SELECTION["cases"])
    _alpha_args.seeds = [int(_seed) for _seed in LEVIR_FTSC_ALPHA_SELECTION["seeds"]]
    _alpha_args.workers = int(LEVIR_FTSC_ALPHA_SELECTION["workers"])
    _alpha_args.device = str(LEVIR_FTSC_ALPHA_SELECTION["device"])
    _alpha_args.allow_alpha6_bridge = bool(LEVIR_FTSC_ALPHA_SELECTION["allow_alpha6_bridge"])
    # Keep the ORIGINAL run namespace across Git bug-fix commits.
    # The old commit-based namespace already has ALPHA4_rerun/seed_42 artifacts.
    # Never compute it again from the new Git SHA, or seed_42 would retrain.
    _alpha_generation = "levir_ftsc_position_alpha_historical_h2_v1_rerun_d3e60b1e59d2"
    _alpha_suite.NAMESPACE = _alpha_generation
    _alpha_args.project = _alpha_root / "runs" / _alpha_generation
    _alpha_args.dataset_root = _alpha_root / "datasets"
    # Dataset download cell materializes 'All Images' and 'All Annotations' in cwd.
    _alpha_data_root = _AlphaPath.cwd()
    if not (_alpha_data_root / "All Images").is_dir():
        _alpha_data_root = _AlphaPath.cwd() / "LevirShipData"
    _alpha_args.data_root = _alpha_data_root.resolve()
    _alpha_args.pretrained = _AlphaPath(yolo_ftsc_pretrained_path).resolve()
    _alpha_args.live_preflight = bool(LEVIR_FTSC_ALPHA_PREFLIGHT_ONLY)
    _alpha_args.confirm_run = not _alpha_args.live_preflight
    _alpha_suite.validate_args(_alpha_args)
    def _alpha_enable_fork_dataloader_workers(_workers, _start_method):
        """Avoid spawn serialization for the patched FTSC dataset transforms."""
        if _workers == 0:
            return
        if _workers < 0:
            raise ValueError(f"workers must be non-negative; received {_workers}")
        if _start_method != "fork":
            raise ValueError(
                "This Linux marimo notebook requires dataloader_start_method='fork' "
                "when workers > 0: spawn cannot pickle the patched transforms"
            )

        import multiprocessing as _ftsc_multiprocessing
        from ultralytics.data import build as _ftsc_data_build

        if "fork" not in _ftsc_multiprocessing.get_all_start_methods():
            raise RuntimeError(
                "This runtime does not support multiprocessing fork; use workers=0"
            )

        _loader_type = _ftsc_data_build.InfiniteDataLoader
        if getattr(_loader_type, "_ftsc_fork_context_enabled", False):
            return

        _original_init = _loader_type.__init__
        _fork_context = _ftsc_multiprocessing.get_context("fork")

        def _init_with_fork_context(_self, *_args, **_kwargs):
            _worker_count = _kwargs.get("num_workers", 0)
            if _worker_count > 0:
                _kwargs.setdefault("multiprocessing_context", _fork_context)
            return _original_init(_self, *_args, **_kwargs)

        _loader_type.__init__ = _init_with_fork_context
        _loader_type._ftsc_fork_context_enabled = True

    _alpha_enable_fork_dataloader_workers(_alpha_args.workers, "fork")
    if _alpha_args.live_preflight:
        LEVIR_FTSC_ALPHA_REPORT = _alpha_suite.run_suite(_alpha_args)
    else:
        if not isinstance(HF_TOKEN, str) or not HF_TOKEN.strip():
            raise RuntimeError("HF_TOKEN is required for immediate result archival")
        _alpha_uploader = _AlphaUploader(
            token=HF_TOKEN, cache_dir=f"/marimo/hf_results_cache/{_alpha_generation}"
        )
        LEVIR_FTSC_ALPHA_REPORT = _alpha_suite.run_suite(_alpha_args, uploader=_alpha_uploader)
    print(LEVIR_FTSC_ALPHA_REPORT)

In [ ]:
# ES1 ORIGINAL-HF METRIC RECOVERY — evaluation only; no training, no HF writes.
# Set True for a dedicated Run All. This automatically skips the Phase-2 launcher below.
RUN_LEVIR_ES1_METRIC_RECOVERY = False
if RUN_LEVIR_ES1_METRIC_RECOVERY:
    import csv as _es1_csv
    import math as _es1_math
    import os as _es1_os
    import statistics as _es1_stats
    import sys as _es1_sys
    from pathlib import Path as _ES1Path

    _es1_root = _ES1Path(yolo_code_dir).resolve()
    for _es1_import_root in (_es1_root, _es1_root / "models_related/ultralytics"):
        if str(_es1_import_root) not in _es1_sys.path:
            _es1_sys.path.insert(0, str(_es1_import_root))
    from utils import ftsc_journal_phase2_one_shot as _es1_phase
    from evaluate_test import standard_detection_metrics as _es1_evaluator
    from huggingface_hub import HfApi as _ES1HfApi

    _es1_token = _es1_os.environ.get("HF_TOKEN", "").strip()
    _es1_phase.require(bool(_es1_token), "HF_TOKEN required to read original ES1 checkpoints")
    _es1_paper, _es1_native = _es1_phase.modules("levir")
    _es1_registry = _es1_phase.history_models("levir")
    _es1_phase.require("ES1_LEGACY" in _es1_registry, "ES1 historical registry missing")
    _es1_config, _es1_aliases = _es1_registry["ES1_LEGACY"]
    _es1_phase.require(_es1_aliases == ("ES1",), "ES1 source alias changed")
    _es1_source_root = (_ES1Path.cwd() if (_ES1Path.cwd() / "All Images").is_dir()
                        else _ES1Path.cwd() / "LevirShipData").resolve()
    _es1_phase.require((_es1_source_root / "All Images").is_dir()
                       and (_es1_source_root / "All Annotations").is_dir(),
                       "LEVIR original images/annotations are unavailable")
    _es1_args = _es1_paper.parse_args([])
    _es1_args.data_root = _es1_source_root
    _es1_args.dataset_root = (_es1_root / "datasets").resolve()
    _es1_args.project = (_es1_root / "runs/levir_es1_original_hf_metric_recovery_v1").resolve()
    _es1_args.pretrained = _es1_root / "yolov8n.pt"
    _es1_args.device = "cuda"
    _es1_args.workers = 4
    _es1_paper.validate_args(_es1_args)

    # Published original seed-level test metrics, NOT values from a later backfill.
    _es1_reference = {
        42: {"AP50": 0.8260294168220195, "mAP50-95": 0.3171464125543804},
        43: {"AP50": 0.8246134330727788, "mAP50-95": 0.32141191051992335},
        44: {"AP50": 0.8198411295359872, "mAP50-95": 0.32037664548773315},
    }
    _es1_revision, _es1_remote_files = _es1_phase._source_files(_ES1HfApi(token=_es1_token))
    _es1_remote_set = set(_es1_remote_files)
    _es1_candidates = {}
    for _es1_seed in (42, 43, 44):
        _es1_found = _es1_phase._historical_case_paths(
            _es1_remote_files, "levir", "ES1_LEGACY", _es1_aliases, _es1_seed)
        _es1_phase.require(len(_es1_found) == 1,
                           f"ES1 seed {_es1_seed}: expected one original best.pt, got {_es1_found}")
        _es1_phase.require(_es1_found[0].startswith("levirship/ES1/"),
                           f"ES1 seed {_es1_seed}: source is not levirship/ES1/: {_es1_found}")
        _es1_candidates[_es1_seed] = _es1_found[0]
    print("[ES1] Exact original checkpoints:", _es1_candidates, flush=True)

    _es1_args.project.mkdir(parents=True, exist_ok=True)
    _es1_cache = _ES1Path("/marimo/hf_results_cache/levir_es1_original_hf_metric_recovery_v1")
    _es1_cache.mkdir(parents=True, exist_ok=True)
    _es1_data_yaml, _ = _es1_phase._prepare_data(
        "levir", _es1_paper, _es1_native, _es1_args, 42)
    _es1_rows = []
    _es1_raw_metrics = []
    for _es1_seed in (42, 43, 44):
        _es1_args.seed = _es1_seed
        _es1_out = _es1_phase._legacy_run(
            "levir", _es1_native, _es1_args, "ES1_LEGACY", _es1_seed,
            _es1_config, _es1_candidates[_es1_seed], _es1_remote_set,
            _es1_revision, _es1_token, _es1_cache, _es1_data_yaml, None)
        _es1_folder = _ES1Path(_es1_out["run_dir"])
        _es1_metrics = _es1_phase._metric_file(_es1_folder, "levir")
        _es1_raw_metrics.append(_es1_metrics)
        _es1_row = {
            "seed": _es1_seed, "checkpoint_sha256": _es1_out["checkpoint_sha256"],
            "hf_revision": _es1_revision, "hf_source": _es1_out["source"],
            "local_eval_dir": str(_es1_folder),
        }
        for _es1_metric, _es1_key in _es1_phase.METRICS["levir"].items():
            _es1_value = float(_es1_metrics[_es1_key])
            _es1_phase.require(_es1_math.isfinite(_es1_value),
                               f"ES1 seed {_es1_seed}: nonfinite {_es1_metric}")
            _es1_row[_es1_metric] = _es1_value
        for _es1_metric in ("AP50", "mAP50-95"):
            _es1_row[f"{_es1_metric}_historical"] = _es1_reference[_es1_seed][_es1_metric]
            _es1_row[f"{_es1_metric}_drift_pp"] = 100 * (
                _es1_row[_es1_metric] - _es1_reference[_es1_seed][_es1_metric])
        _es1_row["historical_drift_status"] = (
            "WITHIN_0P2_PP" if all(
                abs(_es1_row[f"{_es1_metric}_drift_pp"]) <= 0.2
                for _es1_metric in ("AP50", "mAP50-95"))
            else "REVIEW_REQUIRED")
        _es1_rows.append(_es1_row)
        print(f"[ES1] seed {_es1_seed}: AP50={100 * _es1_row['AP50']:.3f}%, "
              f"small={100 * _es1_row['AP50-Small']:.3f}%, "
              f"latency={_es1_row['inference_ms']:.3f} ms, "
              f"drift={_es1_row['historical_drift_status']}", flush=True)

    _es1_phase.require(len({_es1_row["checkpoint_sha256"] for _es1_row in _es1_rows}) == 3,
                       "ES1 seed checkpoints are unexpectedly identical")
    _es1_benchmark_keys = (
        "benchmark/gpu_name", "benchmark/backend", "benchmark/device",
        "benchmark/imgsz", "benchmark/batch", "benchmark/precision")
    _es1_speed_comparable = (
        _es1_evaluator.same_benchmark_fingerprint(_es1_raw_metrics)
        and all(all(_es1_metrics.get(_es1_key) is not None
                    for _es1_key in _es1_benchmark_keys)
                for _es1_metrics in _es1_raw_metrics))
    _es1_summary = {
        "case": "ES1_LEGACY", "source_namespace": "levirship/ES1/",
        "lineage": "ORIGINAL_HF_CHECKPOINT_REEVALUATION_NO_TRAINING",
        "hf_revision": _es1_revision, "seeds": [42, 43, 44],
        "metric_definition": "LEVIR native test; AP50-Small=TinyBenchmark 20-32px; NMS IoU=0.50",
        "latency_comparable": _es1_speed_comparable,
        "historical_drift_status": ("WITHIN_0P2_PP" if all(
            row["historical_drift_status"] == "WITHIN_0P2_PP" for row in _es1_rows)
            else "REVIEW_REQUIRED_DO_NOT_MIX_HISTORICAL_AND_BACKFILL_METRICS"),
    }
    for _es1_metric in _es1_phase.METRICS["levir"]:
        if _es1_metric == "inference_ms" and not _es1_speed_comparable:
            _es1_summary[_es1_metric] = {"status": "BENCHMARK_INCOMPARABLE"}
            continue
        _es1_values = [row[_es1_metric] for row in _es1_rows]
        _es1_summary[_es1_metric] = {
            "mean": _es1_stats.mean(_es1_values),
            "sample_sd": _es1_stats.stdev(_es1_values)}
    _es1_summary["rows"] = _es1_rows
    _es1_phase.atomic_json(_es1_args.project / "es1_original_hf_metrics_3seeds.json", _es1_summary)
    with (_es1_args.project / "es1_original_hf_metrics_by_seed.csv").open(
            "w", newline="", encoding="utf-8") as _es1_stream:
        _es1_writer = _es1_csv.DictWriter(_es1_stream, fieldnames=list(_es1_rows[0]))
        _es1_writer.writeheader()
        _es1_writer.writerows(_es1_rows)
    print("[ES1] Recovery report:", _es1_args.project / "es1_original_hf_metrics_3seeds.json")
    print("[ES1] Historical drift:", _es1_summary["historical_drift_status"], flush=True)


In [ ]:
# LEVIR causal B1: reuse compatible HF checkpoints; retrain absent or incompatible seeds.
# Scope: YOLOv8n P2/P3 FTSC-OFF only; seeds 42, 43, 44.
# Historical FTSC H2, other journal ablations and their results remain untouched.
RUN_LEVIR_V8_P23_PORTABILITY_BACKFILL = True
LEVIR_V8_B1_PREFLIGHT_ONLY = False

if RUN_LEVIR_V8_P23_PORTABILITY_BACKFILL:
    import csv as _b1_csv
    import hashlib as _b1_hashlib
    import json as _b1_json
    import multiprocessing as _b1_mp
    import os as _b1_os
    import shutil as _b1_shutil
    import statistics as _b1_stats
    import sys as _b1_sys
    from pathlib import Path as _B1Path, PurePosixPath as _B1PurePosixPath

    from huggingface_hub import HfApi as _B1HfApi, hf_hub_download as _b1_hf_download

    _b1_root = _B1Path(yolo_code_dir).resolve()
    _b1_vendor = _b1_root / "models_related/ultralytics"
    for _b1_import_path in (str(_b1_root), str(_b1_vendor)):
        if _b1_import_path not in _b1_sys.path:
            _b1_sys.path.insert(0, _b1_import_path)

    from train_levir_scripts import train_levir_v8_p23_portability_backfill_v1 as _b1_contract
    from train_levir_scripts import train_all_levir_yolov8n_p2_ftsc_head_pruning as _b1_reference
    from train_levir_scripts import train_all_levir_yolov8n_p2_routing as _b1_workflow
    from train_levir_scripts import train_levir_ftsc_cue_corrections_v2 as _b1_paper_args
    from evaluate_test.standard_detection_metrics import (
        evaluate_run as _b1_evaluate, load_merged_metrics as _b1_metrics
    )
    from utils.ftsc_journal_phase2_one_shot import ensure_official_pretrained as _b1_check_pretrained
    from utils.ftsc_paper_preflight import layers as _b1_layers, load_yaml as _b1_load_yaml
    from utils.ftsc_results_uploader import FTSCResultsUploader as _B1Uploader
    from ultralytics import YOLO as _B1YOLO

    def _b1_require(condition, message):
        if not condition:
            raise RuntimeError("[LEVIR B1] " + message)

    def _b1_sha(path):
        return _b1_hashlib.sha256(_B1Path(path).read_bytes()).hexdigest()

    # The HF inventory is read BEFORE any data preparation or GPU training.
    _b1_token = _b1_os.environ.get("HF_TOKEN", "").strip()
    _b1_require(bool(_b1_token), "HF_TOKEN required; HF errors must not trigger training")
    _b1_repo = "TLMHoang/FTSC_results"
    _b1_api = _B1HfApi(token=_b1_token)
    _b1_repo_info = _b1_api.repo_info(repo_id=_b1_repo, repo_type="dataset")
    _b1_revision = _b1_repo_info.sha
    _b1_require(isinstance(_b1_revision, str) and len(_b1_revision) >= 30,
                "HF revision not verified")
    _b1_files = set(_b1_api.list_repo_files(
        repo_id=_b1_repo, repo_type="dataset", revision=_b1_revision))
    _b1_require(isinstance(_b1_files, set), "HF file inventory failed")
    _b1_seeds = (42, 43, 44)
    _b1_case = "V8_B1_P23_NOFTSC"
    _b1_remote = {}
    for _b1_seed in _b1_seeds:
        _b1_prefixes = set()
        for _b1_file in _b1_files:
            _b1_parts = _B1PurePosixPath(_b1_file).parts
            for _b1_index in range(1, len(_b1_parts) - 1):
                if (_b1_parts[_b1_index] == _b1_case
                    and _b1_parts[_b1_index + 1] == f"seed_{_b1_seed}"
                    and _b1_parts[0] != "tinyperson"
                    and _b1_parts[0] != "varroa"):
                    _b1_prefixes.add("/".join(_b1_parts[:_b1_index + 2]))
        _b1_preferred_prefix = (
            f"levir_v8_p23_ftsc_off_hf_first_v1/{_b1_case}/seed_{_b1_seed}")
        if _b1_preferred_prefix in _b1_prefixes:
            _b1_prefix = _b1_preferred_prefix
            if len(_b1_prefixes) > 1:
                print(f"[LEVIR B1] seed {_b1_seed}: prefer canonical namespace "
                      f"{_b1_prefix}; ignore legacy copies", flush=True)
        else:
            _b1_require(len(_b1_prefixes) <= 1,
                        f"seed {_b1_seed}: ambiguous legacy HF namespaces: {sorted(_b1_prefixes)}")
            _b1_prefix = next(iter(_b1_prefixes), None)
        if _b1_prefix is not None:
            _b1_require(f"{_b1_prefix}/weights/best.pt" in _b1_files,
                        f"seed {_b1_seed}: partial HF archive without best.pt; manual review required")
        _b1_remote[_b1_seed] = _b1_prefix
    print("[LEVIR B1] Pinned HF inventory:", _b1_revision,
          "checkpoint roots:", _b1_remote, flush=True)
    # No checkpoint means no matching case/seed prefix, NOT an HF connection failure.

    _b1_pretrained = _B1Path(yolo_ftsc_pretrained_path).resolve()
    _b1_check_pretrained(_b1_pretrained)
    _b1_require(_b1_contract.static_graph_parity()["b1_ftsc"] is False,
                "B1 vs historical H2 graph parity failed")
    _b1_config = _b1_contract.CONFIGS[_b1_case]
    _b1_model_yaml = _b1_load_yaml(_b1_config)
    _b1_require(_b1_model_yaml.get("ftsc", {}).get("enabled") is False,
                "B1 YAML does not disable FTSC")

    _b1_args = _b1_paper_args.parse_args([])
    _b1_args.data_root = (_B1Path.cwd() if (_B1Path.cwd() / "All Images").is_dir()
                         else _B1Path.cwd() / "LevirShipData").resolve()
    _b1_args.dataset_root = (_b1_root / "datasets").resolve()
    _b1_args.pretrained = _b1_pretrained
    _b1_args.workers = 4
    _b1_args.device = "cuda"
    _b1_data_yaml = _b1_workflow.prepare_fixed_split(_b1_args)
    _b1_split_manifest = _b1_data_yaml.parent / "manifest.json"
    _b1_require(_b1_split_manifest.is_file(), "Missing fixed-split manifest")
    _b1_split_sha = _b1_sha(_b1_split_manifest)
    _b1_project = _b1_root / "runs/levir_v8_p23_ftsc_off_hf_first_v1" / _b1_case
    _b1_cache = _B1Path("/marimo/hf_results_cache/levir_v8_p23_ftsc_off_hf_first_v1")
    _b1_results = _b1_root / "results"
    _b1_train_template = _b1_reference.train_kwargs(_b1_args, _b1_data_yaml, 42, True)
    _b1_require(_b1_train_template["optimizer"] == "auto" and
                _b1_train_template["mosaic"] == 0.0 and
                _b1_train_template["fitness_metric"] == "map50_95",
                "Historical H2 training protocol changed")

    class _B1CheckpointMismatch(RuntimeError):
        pass

    def _b1_checkpoint_require(condition, message):
        if not condition:
            raise _B1CheckpointMismatch("[LEVIR B1] " + message)

    # Verify the *actual checkpoint* rather than trusting a folder or filename.
    def _b1_verify_checkpoint(path, seed):
        _b1_workflow.local_ultralytics()
        wrapper = _B1YOLO(str(path), task="detect")
        head = wrapper.model.model[-1]
        _b1_checkpoint_require(tuple(head.f) == (19, 22),
                    f"seed {seed}: Detect head mismatch")
        _b1_checkpoint_require(tuple(float(x) for x in head.stride.tolist()) == (4.0, 8.0),
                    f"seed {seed}: P2/P3 stride mismatch")
        _b1_checkpoint_require(not bool(getattr(head, "ftsc_calibrator", None)),
                    f"seed {seed}: checkpoint has FTSC enabled")
        _b1_checkpoint_require(_b1_layers(wrapper.model.yaml) == _b1_layers(_b1_model_yaml),
                    f"seed {seed}: checkpoint architecture mismatch")
        _b1_checkpoint_require(wrapper.model.yaml.get("ftsc", {}).get("enabled", False) is False,
                    f"seed {seed}: checkpoint YAML enables FTSC")
        ckpt_args = wrapper.ckpt.get("train_args") if isinstance(wrapper.ckpt, dict) else None
        _b1_checkpoint_require(isinstance(ckpt_args, dict) and int(ckpt_args.get("seed", -1)) == seed,
                    f"seed {seed}: checkpoint training seed missing/mismatched")
        _b1_expected = _b1_reference.train_kwargs(_b1_args, _b1_data_yaml, seed, True)
        for key in ("epochs", "imgsz", "batch", "patience", "deterministic",
                    "optimizer", "mosaic", "close_mosaic", "lrf", "fitness_metric"):
            _b1_checkpoint_require(key in ckpt_args and ckpt_args[key] == _b1_expected[key],
                        f"seed {seed}: checkpoint training mismatch: {key}")
        return ckpt_args

    def _b1_download(remote):
        return _B1Path(_b1_hf_download(
            repo_id=_b1_repo, repo_type="dataset", filename=remote,
            revision=_b1_revision, token=_b1_token, local_dir=str(_b1_cache)))

    def _b1_manifest(seed):
        kw = _b1_reference.train_kwargs(_b1_args, _b1_data_yaml, seed, True)
        # Runtime-dependent paths/devices are not part of the scientific protocol.
        recipe = {k: v for k, v in kw.items()
                  if k not in ("data", "device", "workers")}
        return {
            "scientific_contract": {
                "schema": "levir_v8_p23_ftsc_off_hf_first_v1",
                "dataset": "LEVIR-Ship", "case": _b1_case, "seed": seed,
                "canonical_seeds": list(_b1_seeds), "split_seed": 42,
                "dataset_manifest_sha256": _b1_split_sha,
                "model_config_sha256": _b1_sha(_b1_config),
                "ftsc_enabled": False, "detect_inputs": [19, 22],
                "pretrained_sha256": _b1_sha(_b1_pretrained),
                "training": recipe,
            },
            "artifact_provenance": {"model_yaml": str(_b1_config),
                                    "pretrained": str(_b1_pretrained)},
        }

    def _b1_guard_local(run_dir, manifest):
        mp = run_dir / "experiment_manifest.json"
        if mp.is_file():
            existing = _b1_json.loads(mp.read_text(encoding="utf-8"))
            _b1_require(existing.get("scientific_contract") == manifest["scientific_contract"],
                        f"Scientific manifest mismatch: {run_dir}")
        else:
            _b1_require(not run_dir.exists() or not any(run_dir.iterdir()),
                        f"Unregistered nonempty output must not be adopted: {run_dir}")
            run_dir.mkdir(parents=True, exist_ok=True)
            mp.write_text(_b1_json.dumps(manifest, indent=2) + "\n", encoding="utf-8")

    def _b1_enable_fork_workers():
        from ultralytics.data import build as _b1_build
        loader = _b1_build.InfiniteDataLoader
        if getattr(loader, "_ftsc_fork_context_enabled", False):
            return
        _b1_require("fork" in _b1_mp.get_all_start_methods(),
                    "Dataloader fork unavailable; do not silently change workers")
        original = loader.__init__
        ctx = _b1_mp.get_context("fork")
        def with_fork(self, *args, **kwargs):
            if kwargs.get("num_workers", 0) > 0:
                kwargs.setdefault("multiprocessing_context", ctx)
            return original(self, *args, **kwargs)
        loader.__init__ = with_fork
        loader._ftsc_fork_context_enabled = True

    _b1_rows = []
    _b1_uploader = None
    for _b1_seed in _b1_seeds:
        _b1_run = _b1_project / f"seed_{_b1_seed}"
        _b1_expected_manifest = _b1_manifest(_b1_seed)
        _b1_prefix = _b1_remote[_b1_seed]
        _b1_action = None

        # Prefer compatible HF checkpoints; only explicit contract mismatches retrain.
        if _b1_prefix is not None:
            _b1_ckpt_remote = _b1_download(f"{_b1_prefix}/weights/best.pt")
            try:
                _b1_verify_checkpoint(_b1_ckpt_remote, _b1_seed)
                _b1_manifest_name = f"{_b1_prefix}/experiment_manifest.json"
                _b1_checkpoint_require(_b1_manifest_name in _b1_files,
                            f"seed {_b1_seed}: HF checkpoint has no split/protocol manifest")
                _b1_source_manifest = _b1_json.loads(
                    _b1_download(_b1_manifest_name).read_text(encoding="utf-8"))
                _b1_source_contract = _b1_source_manifest.get("scientific_contract", {})
                for key, value in (
                    ("case", _b1_case), ("seed", _b1_seed),
                    ("dataset_manifest_sha256", _b1_split_sha),
                    ("model_config_sha256", _b1_sha(_b1_config)),
                    ("ftsc_enabled", False),
                    ("pretrained_sha256", _b1_sha(_b1_pretrained)),
                ):
                    _b1_checkpoint_require(_b1_source_contract.get(key) == value,
                                f"seed {_b1_seed}: HF provenance mismatch: {key}")
                if not LEVIR_V8_B1_PREFLIGHT_ONLY:
                    _b1_guard_local(_b1_run, _b1_expected_manifest)
                    _b1_destination = _b1_run / "weights/best.pt"
                    _b1_destination.parent.mkdir(parents=True, exist_ok=True)
                    if _b1_destination.exists():
                        _b1_require(_b1_sha(_b1_destination) == _b1_sha(_b1_ckpt_remote),
                                    f"seed {_b1_seed}: local/HF checkpoint collision")
                    else:
                        _b1_shutil.copy2(_b1_ckpt_remote, _b1_destination)
                    (_b1_run / "hf_checkpoint_source.json").write_text(
                        _b1_json.dumps({"repo": _b1_repo, "revision": _b1_revision,
                                        "path": f"{_b1_prefix}/weights/best.pt",
                                        "sha256": _b1_sha(_b1_ckpt_remote)}, indent=2) + "\n",
                        encoding="utf-8")
                _b1_action = "HF_REUSED"
            except _B1CheckpointMismatch as _b1_mismatch:
                print(f"[LEVIR B1] HF_REJECTED -> TRAIN seed {_b1_seed}: {_b1_mismatch}", flush=True)
                _b1_action = None
        if _b1_action is None:
            if not LEVIR_V8_B1_PREFLIGHT_ONLY:
                _b1_guard_local(_b1_run, _b1_expected_manifest)
                _b1_best = _b1_run / "weights/best.pt"
                _b1_last = _b1_run / "weights/last.pt"
                if _b1_best.is_file():
                    _b1_verify_checkpoint(_b1_best, _b1_seed)
                    _b1_action = "LOCAL_REUSED"
                else:
                    _b1_workflow.local_ultralytics()
                    _b1_enable_fork_workers()
                    if _b1_last.is_file():
                        _b1_verify_checkpoint(_b1_last, _b1_seed)
                        _B1YOLO(str(_b1_last), task="detect").train(resume=True)
                        _b1_action = "LOCAL_RESUMED"
                    else:
                        _b1_require(not (_b1_run / "results.csv").exists(),
                                    "Unregistered partial training artifacts; refusing overwrite")
                        _b1_workflow.seed_everything(_b1_seed)
                        _b1_model = _B1YOLO(str(_b1_config), task="detect")
                        _b1_model.load(str(_b1_pretrained), smart_transfer=True)
                        _b1_kwargs = _b1_reference.train_kwargs(
                            _b1_args, _b1_data_yaml, _b1_seed, True)
                        _b1_kwargs.update(project=str(_b1_project),
                                          name=f"seed_{_b1_seed}", exist_ok=True)
                        _b1_model.train(**_b1_kwargs)
                        _b1_action = "NEW_TRAINED"
                    _b1_require(_b1_best.is_file(), f"seed {_b1_seed}: missing best.pt after training")
                    _b1_verify_checkpoint(_b1_best, _b1_seed)
            else:
                _b1_action = "HF_MISSING_TRAIN_REQUIRED"

        if LEVIR_V8_B1_PREFLIGHT_ONLY:
            _b1_rows.append({"case": _b1_case, "seed": _b1_seed, "status": _b1_action})
            continue

        _b1_best = _b1_run / "weights/best.pt"
        _b1_require(_b1_best.is_file(), f"seed {_b1_seed}: checkpoint missing")
        _b1_required = _b1_contract.REQUIRED_METRICS
        # Canonical evaluation artifact for this workflow and HF upload.
        _b1_metrics_path = _b1_run / "evaluation_metrics.json"
        _b1_legacy_metrics = _b1_run / "evaluation_metrics_extended.json"
        if not _b1_metrics_path.is_file() and _b1_legacy_metrics.is_file():
            _b1_existing_metrics = _b1_json.loads(_b1_legacy_metrics.read_text(encoding="utf-8"))
            if all(_b1_existing_metrics.get(k) is not None for k in _b1_required):
                _b1_legacy_metrics.replace(_b1_metrics_path)
        _b1_metric_values = (_b1_json.loads(_b1_metrics_path.read_text(encoding="utf-8"))
                             if _b1_metrics_path.is_file() else {})
        _b1_complete_metrics = all(
            _b1_metric_values.get(k) is not None for k in _b1_required)
        if not _b1_complete_metrics:
            _b1_metric_values = _b1_evaluate(
                run_dir=_b1_run, data_yaml=_b1_data_yaml, dataset="levir",
                imgsz=512, batch=8, workers=_b1_args.workers,
                device=_b1_args.device, nms_iou=0.5, include_size=True)
            _b1_require(all(_b1_metric_values.get(k) is not None for k in _b1_required),
                        f"seed {_b1_seed}: incomplete six-core evaluation")
            # The shared evaluator emits the extended filename; normalize its output.
            _b1_legacy_metrics.replace(_b1_metrics_path)
        _b1_require(all(_b1_metric_values.get(k) is not None for k in _b1_required),
                    f"seed {_b1_seed}: incomplete six-core evaluation")
        _b1_rows.append({"case": _b1_case, "seed": _b1_seed, "status": _b1_action,
                         "checkpoint_sha256": _b1_sha(_b1_best), **_b1_metric_values})

        if _b1_action in ("NEW_TRAINED", "LOCAL_RESUMED", "LOCAL_REUSED"):
            if _b1_uploader is None:
                _b1_uploader = _B1Uploader(
                    token=_b1_token, cache_dir=str(_b1_cache))
            _b1_config_copy = _b1_run / "config.yaml"
            if not _b1_config_copy.is_file():
                _b1_shutil.copy2(_b1_config, _b1_config_copy)
            _b1_uploader.publish_run(
                namespace="levir_v8_p23_ftsc_off_hf_first_v1",
                case=_b1_case, seed=_b1_seed, run_dir=_b1_run,
                required_artifacts=(
                    "weights/best.pt", "experiment_manifest.json", "config.yaml",
                    "args.yaml", "results.csv", "evaluation_metrics.json"),
                summary_metadata={"dataset": "LEVIR-Ship", "ftsc_enabled": False,
                                  "source": _b1_action,
                                  "checkpoint_sha256": _b1_sha(_b1_best)},
            )
        print(f"[LEVIR B1] seed {_b1_seed}: {_b1_action}; "
              f"AP50={100*_b1_metric_values['test/metrics/mAP50(B)']:.3f}%, "
              f"AP50-Small={100*_b1_metric_values['test_size/AP50-Small']:.3f}%",
              flush=True)

    if LEVIR_V8_B1_PREFLIGHT_ONLY:
        print("[LEVIR B1] HF-first preflight only:", _b1_rows, flush=True)
    else:
        _b1_require(len(_b1_rows) == 3 and
                    {row["seed"] for row in _b1_rows} == set(_b1_seeds),
                    "Do not publish incomplete three-seed paper aggregates")
        _b1_results.mkdir(parents=True, exist_ok=True)
        _b1_run_csv = _b1_results / "levir_v8_p23_ftsc_off_hf_first_v1_summary_runs.csv"
        _b1_aggregate_csv = _b1_results / "levir_v8_p23_ftsc_off_hf_first_v1_summary_aggregate.csv"
        with _b1_run_csv.open("w", newline="", encoding="utf-8") as out:
            keys = sorted({key for row in _b1_rows for key in row})
            writer = _b1_csv.DictWriter(out, fieldnames=keys)
            writer.writeheader()
            writer.writerows(_b1_rows)
        _b1_aggr = {"case": _b1_case, "n_complete": 3,
                    "seeds": "42,43,44", "hf_revision": _b1_revision}
        for metric in _b1_contract.REQUIRED_METRICS:
            values = [float(row[metric]) for row in _b1_rows]
            _b1_aggr[metric + "/mean"] = _b1_stats.mean(values)
            _b1_aggr[metric + "/std"] = _b1_stats.stdev(values)
        with _b1_aggregate_csv.open("w", newline="", encoding="utf-8") as out:
            writer = _b1_csv.DictWriter(out, fieldnames=list(_b1_aggr))
            writer.writeheader()
            writer.writerow(_b1_aggr)
        print("[LEVIR B1] Complete 3-seed metrics:", _b1_aggregate_csv, flush=True)
